BATCH VS MINI-MATCH VS ON LINE LEARNING

Strategia di batching

Il modo in cui presentiamo i dati alla rete determina se essa imparera con la precisione di uno scienzato o di chi con un approccio frenetico, sbaglia continuamente.

- I 3 paradigmi dell'ottimizzazione: online, mini-batch, batch completo
- Vantaggi del mini-batch, diventato lo standard assoluto, scoprendo il legame tra statistica e potenza della GPU
- Implementazione pratica con configurazione di un DataLoader di Pytorch per gestire diverse dimensioni di batch e carichi di memoria. 

VELOCITA' CONTRO ACCURATEZZA NEL CALCOLO DEL GRADIENTE

Dobbiamo decidere quanti esempi la rete deve guardare prima di cambiare i suoi pesi. Questa scelta definisce il regime di addestramento.
Questa scelta definisce quanto sarà stabile il cammino della rete verso la soluzione.
3 possibi soluzioni:
- Online (Stochastic) Learning: il modello aggiorna i pesi dopo ogni singolo esempio. E' estremamente veloce ma produce un gradiente molto rumoroso. Guardo un dato, campio i pesi, guardo il prossimo, cambio di nuovo, velocissimo nell'iniziare ma il gradiente è rumoroso, come una bussola che oscilla violentemente.
- Batch Learning: il modello calcola il gradiente medio su tutto il dataset prima di aggiornare. E' molto stabile ma richiede enormi quantità di memoria. Guardo tutto il dataset, faccio la media degli errori e solo allora sposto i pesi, è la massima precisione ma richiede una memoria immmensa.
- Mini-Batch Learning: si divide il dataset in piccoli blocchi. E' il compromesso ideale che bilancia la rumorosità stocastica con la stabilità vettoriale. Compromesso ideale, si divide il dataset in piccoli gruppi, solitamente potenze di due come 32, 64, 128. 
Se ho 1000 foto ed un batch di 100 farà 10 iterazioni per completare un'epoca.

Varianza del gradiente
Nel motodo online il gradiente è instabile. Il gradiente di un solo peso può puntare in una direzione opposta a quella dell'ottimo globale a causa di outlier o rumore nei dati.

Stabilità del batch completo
Fornisce una direzione esatta della massima discesa, ma il costo computazionale per calcolare la derivata su milioni di punti è elevato

Il ruolo dello Stochastic Gradient Descent (SGD)
Inciampa e di muove in modo caotico, può aiutare la rete saltare fuori da piccoli vincoli matematici, che invece bloccherebbero un approccio troppo rigido.

Immagina di voler raggiungere il fondo di una valle: il Batch Gradient Descent procede cocon passi lenti e calcolati lungo la linea di massima pendenza. L'online Learnig assomiglia a un ubriaco che inciammpa continuaamente ma che, mediamente, scende verso il basso.
Il mini-batch è come un escursionista che guarda la bussola ogni pochi passi: non è lento come il batch, ma nonn è erratico come l'approccio on line, mantenendo una traiettoria efficiente.

C'è anche una ragione hardware concreta perchè utilizziamo la mini-batch, con la GPU calcolare un gradiente per un solo esempio o per 32 esempi richiede quasi lo stesso tempo perchè la GPU può processare 32 esempi contemporaneamente. Questa tecnica sfrutta la vettorizzazione delle operazioni matriciali, permettendo di ottenere stime del gradiente molto più precise rispetto al caso singolo con un costo temporale quasi identi.
Se ho un dataset di 100 giga e una scheda video di 8, con il mini batch carico solo un gruppo di dati per volta e non tutti.
Il mini-batch non è quindi solo un compromesso statistico, è il modo più efficiente in cui possiamo dare da mangirare alla scheda video.

Con il mini batch calcoliamo la media dei gradienti individuaali, per poi fare la media arritmetica, questa media è molto affidabile del vero gradiente dell'intero dataset.
Ma mano che la dimensione del batch aumenta, la stima del gradiente converge verso il gradiente del dataset, riducendo le oscillazioni della loss function.
Le dimensioni tipiche (32,64,128) sono potenze di due per allinearsi meglio alla gestione delle memoria nei bus delle architettura hardware.
Infine il rumore residuo nel gradiente del mini-batch agisce come una forma di regolarizzazione implicita, aiutando la rete a non generalizzare.

Come si calcola il gradiente di gruppo?
Se si usa un batch troppo grande la rete diventerà molto precisa ma noiosa, rischiando di convergere verso minimi molto stretti che funzionano molto bene sui dati di addestramento ma falliscono nel mondo reale.
Batch piccoli invece tendono a trovare minimi piatti, zone di errore basso che sono più tolleranti alle piccole variazioni dei dati.
Una regola: se raddoppi la dimensione del batch spesso devi raddopiare anche il learning rate, per compensare il fatto che stai facendo meno aggiornamenti ma più stabili.

Pytorch
Pytorch ci semplifica enormemente la gestione del batching attraverso la classe DATALOADER. Questo strumento si occupa di raggruppare i dati im mini-batch li mescola e caricarli in modo efficiente in memoria.
Con questo strumento è possibile passare da un addestramento online a uno mini-batch o completo cambiando solo un parametro di configurazione.
Parametri del DataLoader:
- batch_size: quanti dati ho in un gruppo, inclusi in ogni iterazione di addestramento
- shuffle: rimescola i dati all'inizio di ogni epoca per evitare che la rete impari l'ordine degli esempi.
- num_workers: permette il caricamente dei dati in parallelo usando più core della CPU, eliminando i colli di bottiglia. La CPU prepara il batch successivo mentra la GPU sta ancora calcolando quella attuale, evitando tempi morti.
- drop_last: decide se scartare l'ultimo batch, se la sua dimensione è inferiore a quelle richieste, evitando instabilità nel calcolo della media.


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

# 1. GENERAZIONE DATASET SINTETICO
# Creiamo 1000 punti
#  y = 2x + rumore  --- funzione di y
X = torch.randn(1000, 1)  #X
y = 2 * X + 0.5 * torch.randn(1000, 1)
dataset = TensorDataset(X, y)

def train_with_batch_size(batch_size, epochs=5):
    """Funzione per addestrare il modello con una specifica dimensione di batch."""
    # Definiamo il DataLoader con la dimensione desiderata
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    # Modello lineare semplice
    model = nn.Linear(1, 1)
    criterion = nn.MSELoss()
    optimizer = optim.SGD(model.parameters(), lr=0.01)
    
    loss_history = []
    
    for epoch in range(epochs):
        for inputs, targets in loader:
            optimizer.zero_grad() # Reset dei gradienti
            outputs = model(inputs) # Forward pass
            loss = criterion(outputs, targets) # Calcolo errore
            loss.backward() # Backpropagation
            optimizer.step() # Aggiornamento pesi
            
            # Salviamo la loss per ogni aggiornamento (iterazione)
            loss_history.append(loss.item())
            
    return loss_history

# 2. ESECUZIONE DEI TRE SCENARI
print("Addestramento in corso...")
# Online: batch_size=1
history_online = train_with_batch_size(batch_size=1)
# Mini-batch: batch_size=32
history_mini = train_with_batch_size(batch_size=32)
# Full Batch: batch_size=1000
history_batch = train_with_batch_size(batch_size=1000)

# 3. VISUALIZZAZIONE DEI RISULTATI
plt.figure(figsize=(12, 6))
plt.plot(history_online, label='Online (Batch=1)', alpha=0.4, color='red')
plt.plot(history_mini, label='Mini-Batch (Batch=32)', alpha=0.8, color='green')
plt.plot(history_batch, label='Full Batch (Batch=1000)', linewidth=2, color='blue')

plt.title("Confronto Stabilità Loss per Dimensione Batch")
plt.xlabel("Iterazioni (Aggiornamenti dei pesi)")
plt.ylabel("MSE Loss")
plt.yscale('log') # Scala logaritmica per vedere meglio le differenze
plt.legend()
plt.grid(True, which="both", ls="-", alpha=0.2)
plt.show()

print("Analisi completata. Osserva come la curva blu è liscia, mentre quella rossa è rumorosa.")

Addestramento in corso...


: 

In [4]:
import matplotlib.pyplot as plt
plt.figure(figsize=(12, 6))
plt.plot(history_online, label='Online')
plt.plot(history_mini, label='Mini-Batch')
plt.plot(history_batch, label='Full Batch')
plt.legend()
plt.savefig("test_loss.png")
plt.close()

NameError: name 'history_online' is not defined

<Figure size 1200x600 with 0 Axes>